# 05 — Model 2.1.2, typed flags as a disposition state

**The model in math terms.** M2.1.2 is the mounted M1.2 stack (adopted Both chains, MIX2 routes, partition link) with a parallel disposition node $B$ beside mastery $L$ on every flag-hosting KC — the state view. Flags read $B$; correctness reads both; the habit hurts performance only when the turn triggers it.

**State.** Per KC, the four-cell joint belief

$$\pi_t(l, d) = P(L_t = l,\ B = d \mid H_t), \qquad l, d \in \{0, 1\}$$

Mastery and habit beliefs are its marginals: $P(L{=}1) = \pi(1,1) + \pi(1,0)$, $P(B{=}1) = \pi(1,1) + \pi(0,1)$

**Transition** (between evidence turns): the skill axis drifts within each habit column; the habit axis is frozen (a trait revealed, never changed):

$$\pi^{\text{pre}}(1, d) = \pi(1, d) + \pi(0, d)\,\tau, \qquad \pi^{\text{pre}}(0, d) = \pi(0, d)(1 - \tau)$$

**Emissions.** Flags read $B$ only, one detector pair per flag:

$$P(F_j{=}\text{fired} \mid B{=}1) = v_1^{(j)} \text{ (fitted)}, \qquad P(F_j{=}\text{fired} \mid B{=}0) = v_0 = 0.01 \text{ (pinned)}$$

Correctness reads both coordinates plus the trigger context, through the table $e_{l,d,\text{trig}}$:

$$e_{1,0,\cdot} = 1-s, \quad e_{1,1,\text{no trig}} = 1-s, \quad e_{1,1,\text{trig}} = \max(1-s-\delta,\ g+0.01), \quad e_{0,d,\cdot} = g$$

with $s, g$ inherited from the adopted chains and $\delta$ the fitted trap penalty — the conditional-harm claim: a live habit degrades an otherwise-sound performer only when its trigger is present.

**Trigger context (the trigger-map ruling).** The $\delta$ switch reads the **designed trap schedule** (the instrument's matrix: conjunction Q2; inverse Q3, Q10, Q12; time-axis Q4, Q6; denominator Q7, Q8, Q10, Q12; base-rate Q1, Q8, Q10–Q12). Flag **observations** follow the realized-applicability ruling and update $B$ wherever the annotation presents them, off-schedule included — evidence realized, contexts designed.

**Update** (one Bayes step over the four cells; flag factors reweight columns, the correctness factor rebalances rows):

$$\pi_t(l, d) \propto \pi^{\text{pre}}_t(l, d) \times e_{l,d,\text{trig}}(o) \times \prod_{j \in A_t} P(F_j = f_j \mid B = d)$$

**KC2, the shared disposition.** Inverse and time-axis are two faces of one confused conditioning conception (the Diaz ruling): one $B$, two detectors with separate fitted $v_1$ pairs, the $\delta$ penalty active when either face's trigger is present. Evidence pools across the faces; a hot $B$ raises the other face's fire forecast (cross-prediction). The two-latent KC2 is the registered fallback.

**Estimation** (two-stage, matching M2.1.1's precedent): chains cached; disposition priors $b_0$ per KC from the share of training participants with any fire on that KC's flags (Laplace $\kappa_B = 1$); detector rates $v_1^{(j)}$ as the fire rate among fire-bearers' presented turns, shrunk toward 0.5 ($\kappa_V = 2$); then $w_\text{mix}$ and $\delta$ on sequential stage-two grids jointly with the anchors ($\delta \in [0.05, 0.35]$, floor and ceiling caging the state ordering).

**Prediction** pools the trap-aware act marginals $\sum_{l,d} \pi(l,d)\, e_{l,d,\text{trig}}(\text{correct})$ through the MIX2 routes and the partition link — unlike M2.1.1, the trigger context (though never the flag symbols) enters the forecast, which is exactly the fork: a hot $B$ lowers triggered turns only.

**File layout.**
* The model: `scripts/model_2_1_2.py` (subclasses `Model_1_2_MIX2`)
* Comparisons: **loaded from stored predictions** — `cache/model_1_2_outer_chain/Model_1_2_MIX2/predictions.csv` and `cache/model_2_1_1/Model_2_1_1/predictions.csv`; neither model is re-run here
* The inner-chain cache: `cache/model_1_2_internal_chain/Model_1_2_Internal_Slip_And_Guess/`
* Harness `scripts/evaluator.py`, data `data/data_annotated.csv`, loader `scripts/data.py`
* Saved outputs: `cache/model_2_1_2/Model_2_1_2/` (cell at the bottom)

**Protocol.** 26-fold leave-one-participant-out, predict-before-update, 312 qc targets, qc base rate 0.641, wrong prevalence 0.359, base-rate log-loss 0.6534.

In [1]:
import pandas as pd
import numpy as np
from scripts.data import load_data
from scripts.evaluator import Evaluator, _metrics
from scripts.model_2_1_2 import Model_2_1_2
from scripts.model_1_2_outer_chain import load_internal_chains

DATA = 'data/data_annotated.csv'
CACHE_DIR = 'cache/model_1_2_internal_chain/Model_1_2_Internal_Slip_And_Guess'
M12_PREDS = 'cache/model_1_2_outer_chain/Model_1_2_MIX2/predictions.csv'
M211_PREDS = 'cache/model_2_1_1/Model_2_1_1/predictions.csv'

df = load_data(DATA)
cache = load_internal_chains(CACHE_DIR)
m12 = pd.read_csv(M12_PREDS)
m211 = pd.read_csv(M211_PREDS)
print(len(df), 'rows |', len(cache), 'cached folds |', len(m12), 'and', len(m211), 'stored comparison predictions')

312 rows | 26 cached folds | 312 and 312 stored comparison predictions


## 1. Run
M2.1.2 through the shared harness with the cached inner chains. Both comparisons come from stored predictions, never re-fitted.

In [2]:
ev = Evaluator(Model_2_1_2, df, model_kwargs=dict(n_restarts=3, chain_cache=cache)).run()
preds = ev.predictions
print('done |', int(ev.metrics['n']), 'targets')

done | 312 targets


## 2. Results

### 2.1 Headline metrics against the stored runs
Same references as the earlier notebooks (qc base rate 0.641, wrong prevalence 0.359, base-rate log-loss 0.653).

In [3]:
pd.DataFrame([dict(model='M2.1.2 (state view)', **{k: round(float(v),4) for k,v in ev.metrics.items()}),
              dict(model='M2.1.1 (observation view, stored)', **{k: round(float(v),4) for k,v in _metrics(m211.y_true, m211.p_pred).items()}),
              dict(model='M1.2 (flag-blind, stored)', **{k: round(float(v),4) for k,v in _metrics(m12.y_true, m12.p_pred).items()})]
             ).set_index('model')[['auc','auprc_wrong','bal_acc','log_loss','accuracy','f1','n']]

,auc,auprc_wrong,bal_acc,log_loss,accuracy,f1,n
model,,,,,,,
M2.1.2 (state view),0.6658,0.5651,0.6093,0.6112,0.6955,0.7939,312.0
"M2.1.1 (observation view, stored)",0.6916,0.5920,0.6162,0.6003,0.7019,0.7983,312.0
"M1.2 (flag-blind, stored)",0.6741,0.5747,0.6018,0.6070,0.6859,0.7860,312.0


### 2.2 The fork: state view against observation view
The registered discriminators are the trap-free probe turns (Q5, Q9): the state view predicts a mastered-but-trapped student succeeds where no trigger is present; the observation view, carrying the fire's damage inside lowered mastery, predicts struggle. The tables show the probe turns for the fire-bearing participants, the row-level helps and hurts, and the per-participant AUC deltas.

In [4]:
j = preds.merge(m211, on=['participant_id','question_number'], suffixes=('_s','_o'))
j['good'] = np.where(j.y_true_s == 1, j.p_pred_s - j.p_pred_o, j.p_pred_o - j.p_pred_s)
print(f'rows helped (good > 0.01): {int((j.good > 0.01).sum())} | rows hurt: {int((j.good < -0.01).sum())}')
probes = j[j.question_number.isin([5, 9]) &
           j.participant_id.isin(['P01','P02','P03','P06','P11','P20','P24'])]
probes[['participant_id','question_number','p_pred_o','p_pred_s','y_true_s']].round(3)

rows helped (good > 0.01): 61 | rows hurt: 140


,participant_id,question_number,p_pred_o,p_pred_s,y_true_s
4,P01,5,0.840,0.823,0
8,P01,9,0.841,0.815,1
16,P02,5,0.568,0.771,1
20,P02,9,0.775,0.741,1
28,P03,5,0.611,0.794,0
32,P03,9,0.410,0.495,0
64,P06,5,0.807,0.800,0
68,P06,9,0.517,0.769,1
124,P11,5,0.577,0.773,1
128,P11,9,0.753,0.731,0


In [5]:
rows = []
for pid, g in j.groupby('participant_id'):
    a1 = _metrics(g.y_true_o, g.p_pred_o)['auc']; a2 = _metrics(g.y_true_s, g.p_pred_s)['auc']
    if a1 == a1 and a2 == a2:
        rows.append(dict(participant=pid, m2_1_1=round(a1,3), m2_1_2=round(a2,3), delta=round(a2-a1,3)))
pd.DataFrame(rows).sort_values('delta').set_index('participant')

,m2_1_1,m2_1_2,delta
participant,,,
P03,0.852,0.741,-0.111
P24,0.500,0.406,-0.094
P20,0.861,0.778,-0.083
P01,0.667,0.593,-0.074
P04,0.370,0.333,-0.037
P23,0.741,0.704,-0.037
P05,0.781,0.750,-0.031
P19,0.364,0.364,0.000
P18,0.909,0.909,0.000


### 2.3 Fitted disposition tables, trap penalty, and bridge anchors

In [6]:
v1 = pd.DataFrame([m.v1 for m in ev.fold_models.values()]).mean().round(3)
b0 = pd.DataFrame([m.b0 for m in ev.fold_models.values()]).mean().round(3)
print('detector rates v1, fold means (fire rate given a live disposition):')
display(v1.to_frame('v1'))
print('disposition priors b0, fold means:')
display(b0.to_frame('b0'))
print('trap penalty delta across folds:', sorted(set(m.delta for m in ev.fold_models.values())),
      '| grid floor 0.05')
print('anchors: s0', round(float(np.mean([m.s0 for m in ev.fold_models.values()])),3),
      '| g0', round(float(np.mean([m.g0 for m in ev.fold_models.values()])),3),
      '| censuses 0.077 / 0.061')
print('w_mix fold mean:', round(float(np.mean([m.shape['w_mix'] for m in ev.fold_models.values()])),3))

detector rates v1, fold means (fire rate given a live disposition):


,v1
conjunction,0.500
inverse,0.660
time_axis,0.775
denominator_neglect,0.788
base_rate_neglect,0.445


disposition priors b0, fold means:


,b0
kc1_sample_space,0.037
kc2_conditioning,0.251
kc4_total_probability,0.073
kc5_bayes_update,0.179


trap penalty delta across folds: [0.05] | grid floor 0.05
anchors: s0 0.091 | g0 0.09 | censuses 0.077 / 0.061
w_mix fold mean: 0.442


### 2.4 Confusion matrices
Threshold 0.5, correct as the positive class, the state view beside the stored observation view.

In [7]:
def cmat(p):
    yhat = (p.p_pred >= 0.5).astype(int)
    cm = pd.crosstab(p.y_true.map({1:'actual correct', 0:'actual wrong'}),
                     yhat.map({1:'predicted correct', 0:'predicted wrong'}))
    return cm.reindex(index=['actual correct','actual wrong'],
                      columns=['predicted correct','predicted wrong'], fill_value=0)

print('M2.1.2:'); display(cmat(preds))
print('M2.1.1 (stored):'); display(cmat(m211))

M2.1.2:


p_pred,predicted correct,predicted wrong
y_true,,
actual correct,183,17
actual wrong,78,34


M2.1.1 (stored):


p_pred,predicted correct,predicted wrong
y_true,,
actual correct,184,16
actual wrong,77,35


## 3. Conclusion

* **Pooled prediction goes to the observation view.** M2.1.2 lands at AUC ~0.666, below both M2.1.1 (~0.691) and the flag-blind M1.2 (~0.674): at the qc layer, routing fires through a trigger-gated disposition forecasts this cohort worse than routing them through mastery.
* **But the probes go to the state view, exactly as registered.** The trap-free turns behave as designed: P06's Q9 (machinery intact, habit live, answer right) rises from 0.48 to 0.77; P02 and P11 recover their Q5 (0.58 to 0.77); P06 gains +0.114 AUC and the recoverers flip from M2.1.1's losers to M2.1.2's winners. The mastered-but-trapped cell exists and earns its keep on the turns built to test it.
* **The two views are near-exact complements on the Q5 battlefield.** The persisters M2.1.1 caught there (P20, P24, P03, wrong answers anticipated through crashed mastery) are un-caught by M2.1.2, whose delta is off on the trap-free turn by construction; their deltas mirror almost perfectly (P24 -0.125, P03 -0.111, P20 -0.083 against P06 +0.114, P11 +0.031). One mechanism's gains are the other's losses, row for row.
* **The reading underneath, worth keeping.** The persisters' Q5 wrongs are largely construal-family failures (world-slips, misreads), not conditioning-habit failures: M2.1.1's wins there were the indiscriminate mastery-crash accidentally covering the misinterpretation channel's territory. The state view is the more honest mechanism and pays for its honesty wherever the true failure mode is one the flags never typed.
* **The trap penalty fits at its floor.** Delta lands at 0.05 in every fold: given the disposition machinery, the qc likelihood wants only minimal conditional harm, so the conditional-harm claim gets little support at this layer, and the state view's value concentrates in not over-penalizing rather than in warning.
* **The bridge holds.** Anchors at s0 ~0.091, g0 ~0.090 against censuses 0.077 and 0.061, slightly looser than M2.1.1's 0.080 but census-adjacent; the fork does not disturb the bridge's meaning.
* **Fork verdict (descriptive, as registered).** Observation view for prediction, state view for representation: M2.1.1 carries the primary contrast; M2.1.2's probe behavior and the recoverer fix are the argument that the disposition is real but that its *performance* consequence in this no-feedback hour is small, which is the question M2.2's gate takes up next. Caveats: participant-clustered intervals deferred until all models are built; ~20 discrimination turns make the probe comparison descriptive at this n.

## 4. Save
Persist the run: per-fold bridge, shape, detector rates, and disposition priors, the pooled predictions, the metrics, and the index.

In [8]:
import os
import json
from scripts.model_2_1_2 import save_model_2_1_2_from_evaluator

out_dir = 'cache/model_2_1_2'
save_model_2_1_2_from_evaluator(ev, out_dir)

'cache/model_2_1_2/Model_2_1_2'